In [1]:
from pyspark.sql import functions as F, Window

AREA_TESELA_M2 = 307 * 307    # cada tesela cubre unos 307 x 307 m

det = spark.table("Lh_Silver.silver_detecciones")
# Por si alguna tesela se procesó dos veces, me quedo con la detección más reciente
w = Window.partitionBy("anio", "tesela").orderBy(F.col("fecha_deteccion").desc())
det = det.withColumn("n", F.row_number().over(w)).filter("n = 1").drop("n")

# Coordenadas de cada tesela, del inventario de Bronze
teselas = spark.table("Lh_Bronze.bronze_teselas").select(
    "anio", "tesela", "lat_min", "lat_max", "lon_min", "lon_max")

StatementMeta(, 73472067-c91f-4c34-be05-0e44ebd81027, 3, Finished, Available, Finished, False)

In [2]:
# Nombres legibles para el informe. El número delante hace que la leyenda
# salga en el orden en que avanza la obra
nombres_fase = {
    "terreno_sin_tocar": " 1. Sin tocar",
    "movimiento_de_tierras": "2. Movimiento de tierras",
    "estructuras_sin_paneles": "3. Estructuras",
    "paneles_instalados": "4. Paneles instalados",
}
mapa_nombres = F.create_map([F.lit(x) for par in nombres_fase.items() for x in par])

gold_teselas = (det.join(teselas, ["anio", "tesela"])
    .withColumn("lat_centro", (F.col("lat_min") + F.col("lat_max")) / 2)
    .withColumn("lon_centro", (F.col("lon_min") + F.col("lon_max")) / 2)
    .withColumn("superficie_paneles_m2", F.col("pct_paneles") / 100 * AREA_TESELA_M2)
    .withColumn("fase_texto", mapa_nombres[F.col("fase")])
    .select("planta", "anio", "tesela", "fila", "columna", "fase", "fase_texto",
            "pct_paneles", "superficie_paneles_m2", "lat_centro", "lon_centro"))

# overwriteSchema porque la tabla gana una columna nueva (fase_texto)
(gold_teselas.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("Lh_Gold.gold_teselas"))
print("gold_teselas:", gold_teselas.count(), "filas")

StatementMeta(, 73472067-c91f-4c34-be05-0e44ebd81027, 4, Finished, Available, Finished, False)

gold_teselas: 672 filas


In [3]:
# Zona de la planta = teselas con paneles en el último año disponible.
# En un proyecto real saldría del plano de implantación de la planta.
ultimo_anio = gold_teselas.agg(F.max("anio")).first()[0]
zona_planta = (gold_teselas
    .filter((F.col("anio") == ultimo_anio) & (F.col("pct_paneles") >= 5))
    .select("tesela"))

en_planta = gold_teselas.join(zona_planta, "tesela")

# Superficie de paneles de la planta terminada (referencia = 100 %)
sup_final = (en_planta.filter(F.col("anio") == ultimo_anio)
             .agg(F.sum("superficie_paneles_m2")).first()[0])

fase = lambda nombre: F.sum(F.when(F.col("fase") == nombre, 1).otherwise(0))

avance = (en_planta.groupBy("planta", "anio").agg(
        F.count("*").alias("teselas_planta"),
        F.sum("superficie_paneles_m2").alias("superficie_paneles_m2"),
        fase("terreno_sin_tocar").alias("teselas_sin_tocar"),
        fase("movimiento_de_tierras").alias("teselas_mov_tierras"),
        fase("estructuras_sin_paneles").alias("teselas_estructuras"),
        fase("paneles_instalados").alias("teselas_paneles"))
    .withColumn("hectareas_paneles", F.round(F.col("superficie_paneles_m2") / 10_000, 1))
    .withColumn("pct_avance", F.round(F.col("superficie_paneles_m2") / sup_final * 100, 1))
    .drop("superficie_paneles_m2")
    .orderBy("anio"))

avance.write.format("delta").mode("overwrite").saveAsTable("Lh_Gold.gold_avance_anual")
display(avance)

StatementMeta(, 73472067-c91f-4c34-be05-0e44ebd81027, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 86405069-1dd3-4258-9b98-afb410918fb9)

In [4]:
# Datos del proyecto (planificación). En una obra real vendrían del sistema de
# gestión del proyecto; aquí uso los datos públicos de Iberdrola
PANELES_TOTALES = 1_430_000
POTENCIA_TOTAL_MWP = 500.0
HA_TESELA = AREA_TESELA_M2 / 10_000    # ~9,4 ha por tesela

gold_paneles = (spark.table("Lh_Gold.gold_avance_anual")
    # Paneles estimados a partir del % de avance (más fiable que la superficie absoluta)
    .withColumn("paneles_totales", F.lit(PANELES_TOTALES))
    .withColumn("paneles_instalados_est",
                F.round(F.col("pct_avance") / 100 * PANELES_TOTALES).cast("bigint"))
    .withColumn("paneles_pendientes_est",
                F.col("paneles_totales") - F.col("paneles_instalados_est"))
    # Potencia instalada estimada
    .withColumn("potencia_total_mwp", F.lit(POTENCIA_TOTAL_MWP))
    .withColumn("potencia_instalada_mwp_est",
                F.round(F.col("pct_avance") / 100 * POTENCIA_TOTAL_MWP, 1))
    # Superficie con obra en marcha (movimiento de tierras + estructuras) y sin empezar
    .withColumn("ha_en_obra",
                F.round((F.col("teselas_mov_tierras") + F.col("teselas_estructuras")) * HA_TESELA, 1))
    .withColumn("ha_sin_empezar", F.round(F.col("teselas_sin_tocar") * HA_TESELA, 1))
    .select("planta", "anio", "pct_avance", "paneles_totales", "paneles_instalados_est",
            "paneles_pendientes_est", "potencia_total_mwp", "potencia_instalada_mwp_est",
            "ha_en_obra", "ha_sin_empezar"))

gold_paneles.write.format("delta").mode("overwrite").saveAsTable("Lh_Gold.gold_avance_paneles")
display(gold_paneles)

StatementMeta(, 73472067-c91f-4c34-be05-0e44ebd81027, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cb466c45-2fcb-4c3d-881c-d94651e8d43a)